In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
# Step 1: Load and clean job description
df = pd.read_csv("C:/Users/HP/Desktop/Internship Final Submission/NLP/2_raw_jobs.csv").dropna(subset=["job_description"]).reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s\+\#]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["job_description"].apply(clean_text)
print(df)

     job_id                         job_title                    company  \
0    100001          Supply Chain Coordinator            Orbital Systems   
1    100002         Backend Software Engineer  Vertex Financial Services   
2    100003                    UX/UI Designer            Lumen Insurance   
3    100004                    Data Scientist       Falconview Aerospace   
4    100005                    Data Scientist             Amber Foods Co   
..      ...                               ...                        ...   
995  100996          Supply Chain Coordinator        Northgate Logistics   
996  100997  Human Resources Business Partner           Crestview Realty   
997  100998                Frontend Developer           Redwood Robotics   
998  100999          Supply Chain Coordinator  Vertex Financial Services   
999  101000  Human Resources Business Partner           Anchor Bank Corp   

                       location  \
0    Mumbai, Maharashtra, India   
1        San Fran

In [3]:
#Step 2: Build word embeddings FROM YOUR OWN CORPUS via LSA 
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english", min_df=3)
X = vectorizer.fit_transform(df["clean_description"])          # documents x terms
vocab = vectorizer.get_feature_names_out()
word_to_idx = {w: i for i, w in enumerate(vocab)}

svd = TruncatedSVD(n_components=100, random_state=42)
word_vectors = svd.fit_transform(X.T)                            # terms x latent dims
print("Word vectors shape:", word_vectors.shape)

Word vectors shape: (396, 100)


In [4]:
# Step 3: Helper functions
def embed(phrase):
    """Average word vectors for a phrase. Returns None if no words are in vocabulary."""
    words = re.sub(r"[^a-z0-9\s]", " ", str(phrase).lower()).split()
    idxs = [word_to_idx[w] for w in words if w in word_to_idx]
    if not idxs:
        return None
    return np.mean(word_vectors[idxs], axis=0)

def similarity(a, b):
    ea, eb = embed(a), embed(b)
    if ea is None or eb is None:
        return None
    return float(cosine_similarity([ea], [eb])[0][0])

In [5]:
# Step 4: Example 
print("ML vs Machine Learning:", similarity("ml", "machine learning"))
print("python vs programming:", similarity("python", "programming"))
print("aws vs cloud:", similarity("aws", "cloud"))

ML vs Machine Learning: None
python vs programming: None
aws vs cloud: 0.8835694732246973


In [6]:
#Step 5: Skill dictionary 
skill_dictionary = [
    "python", "sql", "machine learning", "deep learning", "power bi", "aws",
    "natural language processing", "data analysis", "cloud computing",
    "excel", "tableau", "docker", "kubernetes",
]
skill_embeddings = {s: embed(s) for s in skill_dictionary}
skill_embeddings = {k: v for k, v in skill_embeddings.items() if v is not None}
skill_names = list(skill_embeddings.keys())
skill_matrix = np.array([skill_embeddings[k] for k in skill_names])

def best_skill_match(candidate):
    ec = embed(candidate)
    if ec is None:
        return None, None
    sims = cosine_similarity([ec], skill_matrix)[0]
    idx = sims.argmax()
    return skill_names[idx], float(sims[idx])
print(f"{len(skill_names)}/{len(skill_dictionary)} dictionary skills embedded successfully.")

12/13 dictionary skills embedded successfully.


In [7]:
# Test on abbreviations 
test_candidates = ["ml", "nlp", "aws cloud", "spreadsheets", "stakeholder management", "tensorflow"]
for c in test_candidates:
    skill, score = best_skill_match(c)
    if skill is None:
        print(f"{c!r:30s} -> OUT OF VOCABULARY (no embedding found)")
    else:
        print(f"{c!r:30s} -> best match: {skill!r} (similarity={score:.3f})")

'ml'                           -> OUT OF VOCABULARY (no embedding found)
'nlp'                          -> OUT OF VOCABULARY (no embedding found)
'aws cloud'                    -> best match: 'aws' (similarity=0.978)
'spreadsheets'                 -> OUT OF VOCABULARY (no embedding found)
'stakeholder management'       -> best match: 'data analysis' (similarity=0.316)
'tensorflow'                   -> best match: 'deep learning' (similarity=0.720)


In [8]:
# Test on abbreviations
for c in ["ml", "nlp", "aws cloud", "stakeholder management"]:
    skill, score = best_skill_match(c)
    if skill is None:
        print(f"{c!r:28s} -> OUT OF VOCABULARY")
    else:
        print(f"{c!r:28s} -> best match: {skill!r} (similarity={score:.3f})")

'ml'                         -> OUT OF VOCABULARY
'nlp'                        -> OUT OF VOCABULARY
'aws cloud'                  -> best match: 'aws' (similarity=0.978)
'stakeholder management'     -> best match: 'data analysis' (similarity=0.316)


In [9]:
vec = CountVectorizer(ngram_range=(1, 2), stop_words="english", max_features=300)
X = vec.fit_transform(df["clean_description"])
terms = vec.get_feature_names_out()
counts = np.asarray(X.sum(axis=0)).flatten()
candidates = (
    pd.DataFrame({"term": terms, "count": counts})
    .sort_values("count", ascending=False)
    .head(150)["term"]
    .tolist()
)
print(f"Extracted {len(candidates)} candidate terms from the corpus.")

Extracted 150 candidate terms from the corpus.


In [10]:
rows = []
for c in candidates:
    skill, score = best_skill_match(c)
    if skill is not None:
        rows.append((c, skill, score))

matches = pd.DataFrame(rows, columns=["candidate_term", "best_matching_skill", "similarity"])
matches = matches.sort_values("similarity", ascending=False).reset_index(drop=True)
matches.head(20)

,candidate_term,best_matching_skill,similarity
0,excel,excel,1.000000
1,sql,sql,1.000000
2,power bi,power bi,1.000000
3,bi,power bi,1.000000
4,power,power bi,1.000000
5,kubernetes,kubernetes,1.000000
6,learning,machine learning,0.936463
7,data,data analysis,0.871864
8,engineer,kubernetes,0.856392
9,continuous learning,deep learning,0.834500


In [11]:
non_exact = matches[~matches["candidate_term"].isin(skill_dictionary)]
non_exact.head(20)

,candidate_term,best_matching_skill,similarity
3,bi,power bi,1.000000
4,power,power bi,1.000000
6,learning,machine learning,0.936463
7,data,data analysis,0.871864
8,engineer,kubernetes,0.856392
9,continuous learning,deep learning,0.834500
10,learning provide,deep learning,0.834500
11,analysis,data analysis,0.833953
12,analyst,excel,0.711941
13,models,machine learning,0.686602
